In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as func
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.appName("Superhero").getOrCreate()

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType())
])

names = spark.read.schema(schema).option("sep", " ").csv("./MarvelNames.txt")
lines = spark.read.text("./MarvelGraph.txt")

connections = (
    lines.withColumn("id", func.split(func.column("value"), " ")[0])
    .withColumn("connections", func.size(func.split(func.col("value"), " ")) - 1)
    .groupBy("id").agg(func.sum("connections").alias("connections"))
)

mostPopular = connections.sort(func.col("connections").desc()).first()
mostPopularName = names.filter(func.col("id") == mostPopular[0]).select("name").first()

print(mostPopularName[0] + " is the most popular hero, with " + str(mostPopular[1]) + " appearances!")

# Find the most obscure heroes
mostObscureValue = connections.agg(func.min("connections")).first()[0]
mostObscureIDs = connections.filter(func.col("connections") == mostObscureValue)
mostObscureNames = names.join(mostObscureIDs, "id")

mostObscureNames.show()

spark.stop()